# BEAR Motor Scale Torque Test

Drives one **Koala BEAR Muscle Build (KBMB02)** into a scale/force gauge to characterize its
output torque as a function of commanded `iq` (quadrature current).

**Test flow:**
1. Record the current shaft angle as the **start position** (home) and compute a tight
   `limit_position_min/max` window around the expected ~91° sweep to the scale.
2. Drive the ramp in **Direct Force mode (mode 3)** with its spring gains set to zero, so it
   behaves exactly like Torque mode — `goal_iq` is the only thing moving the shaft — but unlike
   real Torque mode it gets **hardware-enforced joint limits**: per the SDK manual (`BEAR_SDK.pdf`,
   §2.2.2/§2.3), `limit_position_min/max` is only active in modes 2 (Position) and 3 (Direct
   Force), not mode 0 (Torque) or 1 (Velocity). Ramp `goal_iq` up in small steps; the horn/lever
   sweeps roughly 90° and presses ~1° further into the scale to guarantee firm contact.
3. Hold the final `iq` for a set number of seconds so you can read the scale.
4. Ramp `iq` back to 0 and **retract** back to the start position using Position mode.

**Hardware assumptions** (edit the connect cell if different):
- 1 motor, driving a horn/lever into a scale or force gauge mounted ~90° away from home
- USB2BEAR on `/dev/ttyUSB0`-style port at `8 Mbps`
- Power: 3–8S Li-Po within 9–33.6 V

**Safety notes:**
- Two independent layers protect against a missing/mispositioned scale: a software check
  (`POSITION_MARGIN_DEG`) aborts the ramp as soon as the shaft travels past the expected
  contact point, and a hardware backstop (`limit_position_min/max`, a bit further out) puts the
  BEAR into safe damping and raises a Joint Limit error if the software check is somehow too
  slow. Neither exists in true Torque mode, which is why this file uses Direct Force mode
  instead — see section 7.
- `limit_i_max` is the actual torque ceiling and **is** enforced in every mode, including
  Torque and Direct Force.
- Every mode switch goes *disable → set goals → set mode → enable* via the same `switch_mode`
  helper as the base test notebook, so Velocity/Position/Direct-Force are all available too —
  see the appendix at the end.
- Run the *EMERGENCY DISABLE* cell any time the motor misbehaves.

## 1. Connect

In [ ]:
import time
import math
from pybear import Manager

# Edit these if your setup differs from the defaults.
PORT = '/dev/serial/by-id/usb-Westwood_Robotics_USB2BEAR_Boosted_UB0104-if00-port0'
MOTOR_ID = 1
BAUD = 8000000  # USB2BEAR fast setting

bear = Manager.BEAR(
    port=PORT,
    baudrate=BAUD,
    single_timeout=0.01,   # 10 ms — comfortably above FTDI worst case
    single_try_num=3,
    bulk_timeout=0.01,
    bulk_try_num=3,
)

## 2. Ping & helpers

Confirm the motor answers, then define the small helpers reused throughout: `read1` flattens a
single-motor single-register read, and `switch_mode` performs the safe
*disable → set goals → set mode → enable* sequence for any of the four modes.

In [ ]:
ping_data, err = bear.ping(MOTOR_ID)[0]
if err is None:
    raise RuntimeError(f"BEAR {MOTOR_ID} did not respond. Check power, port, baud, and ID.")
print(f"BEAR {MOTOR_ID} online. fw/hw version blob = {ping_data[0]:#x}")
print(f"Error byte = 0x{err:02x} ({bear.decode_error(err)})")

In [ ]:
def read1(getter, motor_id):
    """Flatten [([val], err)] -> (val, err) for single-motor single-reg reads."""
    data_list, err = getter(motor_id)[0]
    return data_list[0], err

MODE_TORQUE   = 0
MODE_VELOCITY = 1
MODE_POSITION = 2
MODE_FORCE    = 3   # "Direct Force" — combined position/velocity/torque

# PyBEAR writes (including set_mode, set_torque_enable) are fire-and-forget: no ack, no
# retry, unlike reads. On a spotty link a set_mode() can silently drop, leaving the motor
# in the OLD mode while goal_iq is driven as if the new mode had taken effect — if the old
# mode was Position/Direct Force, its limit_position_min/max stays enforced, which is how
# a "Torque mode" run can trip a Joint Limit error that mode 0 should never trigger. So
# switch_mode/disable read back (reads DO retry/verify) and resend until confirmed.
VERIFY_TRIES = 5
VERIFY_DELAY = 0.01

def disable():
    """Idempotent. Safe to call any time. Confirms torque_enable reads back 0 before
    returning — clears latched errors (e.g. Joint Limit), and a dropped disable write
    can't silently leave the latch un-cleared."""
    for attempt in range(VERIFY_TRIES):
        bear.set_torque_enable((MOTOR_ID, 0))
        state, err = read1(bear.get_torque_enable, MOTOR_ID)
        if err is not None and state == 0:
            return
        print(f"  [disable] not confirmed yet (state={state}), retrying ({attempt + 1}/{VERIFY_TRIES})...")
        time.sleep(VERIFY_DELAY)
    raise RuntimeError(f"Disable never confirmed after {VERIFY_TRIES} tries — check the link, or power-cycle the motor.")

def switch_mode(motor_id, new_mode, goals: dict):
    """Disable, prime goals for the new mode, switch mode, re-enable.

    `goals` keys can be any of: 'goal_position', 'goal_velocity', 'goal_iq', 'goal_id'.
    """
    disable()
    time.sleep(0.01)  # let the motor coast / damping kick in before we rewrite goals
    setters = {
        'goal_position': bear.set_goal_position,
        'goal_velocity': bear.set_goal_velocity,
        'goal_iq':       bear.set_goal_iq,
        'goal_id':       bear.set_goal_id,
    }
    for name, value in goals.items():
        setters[name]((motor_id, value))

    for attempt in range(VERIFY_TRIES):
        bear.set_mode((motor_id, new_mode))
        actual_mode, err = read1(bear.get_mode, motor_id)
        if err is not None and actual_mode == new_mode:
            break
        print(f"  [switch_mode] mode not confirmed yet (read {actual_mode}), retrying ({attempt + 1}/{VERIFY_TRIES})...")
        time.sleep(VERIFY_DELAY)
    else:
        raise RuntimeError(f"set_mode({new_mode}) never confirmed after {VERIFY_TRIES} tries — check the link before enabling.")

    bear.set_torque_enable((motor_id, 1))

print("Helpers defined.")

## 3. Health snapshot

Bus voltage, three temperatures, and the current shaft position — read before enabling
anything.

In [ ]:
v_bus, _   = read1(bear.get_input_voltage, MOTOR_ID)
t_wind, _  = read1(bear.get_winding_temperature, MOTOR_ID)
t_power, _ = read1(bear.get_powerstage_temperature, MOTOR_ID)
t_ic, _    = read1(bear.get_ic_temperature, MOTOR_ID)
pos0, _    = read1(bear.get_present_position, MOTOR_ID)
torque_en, _ = read1(bear.get_torque_enable, MOTOR_ID)

print(f"Bus voltage      : {v_bus:.2f} V        (KBMB02 wants 9–33.6 V)")
print(f"Winding temp     : {t_wind:.1f} °C")
print(f"Powerstage temp  : {t_power:.1f} °C")
print(f"IC temp          : {t_ic:.1f} °C")
print(f"Present position : {pos0:.4f} rad")
print(f"torque_enable    : {torque_en}  (0 disabled, 1 enabled, 2 critical err, 3 e-stop/damping)")

## 4. Scale test parameters

`DIRECTION` is +1 or -1 depending on which way the horn needs to turn to reach the scale.
`POSITION_MARGIN_DEG` is how far past the expected contact point (`ROTATION_DEG +
OVERTRAVEL_DEG`) the shaft may travel before the *software* check aborts the ramp.
`HARDWARE_LIMIT_PAD_DEG` adds extra room beyond that for the *hardware* `limit_position_max`
(set in section 6) — it should only ever trip if the software check somehow misses it. Start
`IQ_MAX` low and raise it between runs once you've confirmed both checks work and the scale is
reading sensibly.

In [ ]:
ROTATION_DEG            = 90.0   # nominal sweep to reach the scale
OVERTRAVEL_DEG          = 1.0    # extra travel to dig in / guarantee firm contact
DIRECTION               = 1      # +1 or -1 — whichever way drives the horn into the scale
POSITION_MARGIN_DEG     = 5.0    # software abort past the expected contact point
HARDWARE_LIMIT_PAD_DEG  = 3.0    # extra pad beyond POSITION_MARGIN_DEG for the hardware limit
START_SIDE_PAD_DEG      = 2.0    # slack on the near/start side of the hardware limit window

IQ_START   = 0.0    # A
IQ_MAX     = 1.0    # A — raise cautiously across runs; must stay <= limit_i_max
IQ_STEP    = 0.05   # A per ramp step
STEP_DT    = 0.15   # s between ramp steps

HOLD_TIME_S    = 3.0   # s to hold at IQ_MAX so you can read the scale
RETRACT_TIME_S = 1.5   # s for the smooth Position-mode return to start

print(f"Ramp: {IQ_START} A -> {IQ_MAX} A in steps of {IQ_STEP} A, {STEP_DT}s apart "
      f"({int((IQ_MAX - IQ_START) / IQ_STEP)} steps, "
      f"~{(IQ_MAX - IQ_START) / IQ_STEP * STEP_DT:.1f}s ramp time).")

## 5. Capture start position & compute hardware limits

Read the present position while disabled and use it as the home reference. From it, compute
the expected scale-contact position plus a `limit_position_min/max` window: the far edge sits
`ROTATION_DEG + OVERTRAVEL_DEG + POSITION_MARGIN_DEG + HARDWARE_LIMIT_PAD_DEG` degrees out, the
near edge just `START_SIDE_PAD_DEG` degrees behind the start. These limits are written to the
motor in section 6 and are what give Direct Force mode real hardware joint-limit protection.

In [ ]:
START_POS, _ = read1(bear.get_present_position, MOTOR_ID)
EXPECTED_CONTACT_POS = START_POS + DIRECTION * math.radians(ROTATION_DEG + OVERTRAVEL_DEG)

_far_bound  = START_POS + DIRECTION * math.radians(
    ROTATION_DEG + OVERTRAVEL_DEG + POSITION_MARGIN_DEG + HARDWARE_LIMIT_PAD_DEG)
_near_bound = START_POS - DIRECTION * math.radians(START_SIDE_PAD_DEG)
LIMIT_POSITION_MIN = min(_near_bound, _far_bound)
LIMIT_POSITION_MAX = max(_near_bound, _far_bound)

print(f"Start position            : {START_POS:.4f} rad")
print(f"Expected scale-contact pos: {EXPECTED_CONTACT_POS:.4f} rad "
      f"({ROTATION_DEG + OVERTRAVEL_DEG:.1f} deg from start, direction {DIRECTION:+d})")
print(f"Software abort threshold  : {ROTATION_DEG + OVERTRAVEL_DEG + POSITION_MARGIN_DEG:.1f} deg from start")
print(f"Hardware limit window     : [{LIMIT_POSITION_MIN:.4f}, {LIMIT_POSITION_MAX:.4f}] rad")

## 6. One-time configuration (motor must be disabled)

Current-loop PID (KBMB02 recommended values), plus position/velocity loop gains used later for
the retract move. `p/i/d_gain_force` are left at **zero** — Direct Force mode sums a
position/velocity spring term with the `goal_iq` feed-forward, and zeroing the spring gains
means only the feed-forward term acts, so the ramp behaves like pure current control while
still getting hardware position-limit enforcement (spring gains don't affect whether
`limit_position_min/max` is enforced — that happens regardless, per the SDK manual).

`limit_i_max` is the **hard safety ceiling** enforced by the motor firmware in every mode — keep
it comfortably above `IQ_MAX` from section 4 but well under anything that could damage the
horn/scale mount. `limit_position_min/max` use the window computed in section 5. We do not call
`save_config`, so none of this is written to flash — it all resets on power cycle.

In [ ]:
bear.set_torque_enable((MOTOR_ID, 0))

# --- Current loop (id and iq) — recommended for Koala Muscle Build V2 ---
bear.set_p_gain_id((MOTOR_ID, 0.358))
bear.set_i_gain_id((MOTOR_ID, 0.045))
bear.set_d_gain_id((MOTOR_ID, 0.0))
bear.set_p_gain_iq((MOTOR_ID, 0.358))
bear.set_i_gain_iq((MOTOR_ID, 0.045))
bear.set_d_gain_iq((MOTOR_ID, 0.0))

# --- Position/velocity loop — used for the retract move in section 8 ---
bear.set_p_gain_position((MOTOR_ID, 5.0))
bear.set_i_gain_position((MOTOR_ID, 0.0))
bear.set_d_gain_position((MOTOR_ID, 0.2))
bear.set_p_gain_velocity((MOTOR_ID, 0.5))
bear.set_i_gain_velocity((MOTOR_ID, 0.0))
bear.set_d_gain_velocity((MOTOR_ID, 0.0))

# --- Direct Force loop — zeroed so the ramp in section 7 is pure iq feed-forward ---
bear.set_p_gain_force((MOTOR_ID, 0.0))
bear.set_i_gain_force((MOTOR_ID, 0.0))
bear.set_d_gain_force((MOTOR_ID, 0.0))

# --- Safety limits ---
# Hard ceiling on current in every mode, including Torque and Direct Force. KBMB02 KT = 1.16 Nm/A.
bear.set_limit_i_max((MOTOR_ID, 3.0))
# Applies in Velocity and Position modes only.
bear.set_limit_velocity_max((MOTOR_ID, 5.0))
# Applies in Position and Direct Force modes only (NOT Torque or Velocity) — the hardware
# backstop for the scale ramp in section 7. Window computed in section 5.
bear.set_limit_position_min((MOTOR_ID, LIMIT_POSITION_MIN))
bear.set_limit_position_max((MOTOR_ID, LIMIT_POSITION_MAX))

print("Configured. Motor still disabled.")

## 7. Direct-Force-mode ramp into the scale

Enable Direct Force mode anchored at `START_POS` with `goal_iq = 0`. Because the force-loop
gains are zero (section 6), `goal_position`/`goal_velocity` contribute nothing — only `goal_iq`
drives the shaft, exactly like Torque mode. Step `goal_iq` up to `IQ_MAX`, logging commanded/
measured `iq` and position at each step, and checking two independent abort conditions:

- **Software:** shaft has traveled past the expected contact point by more than
  `POSITION_MARGIN_DEG`.
- **Hardware:** `torque_enable` reads back `3` (BEAR entered safe damping because
  `limit_position_min/max` was exceeded — the Joint Limit error).

Either one zeroes `iq` and stops the ramp. Once the ramp completes cleanly, hold at `IQ_MAX`
for `HOLD_TIME_S` seconds — this is your window to read the scale — then ramp back down.

In [ ]:
switch_mode(MOTOR_ID, MODE_FORCE, {'goal_position': START_POS, 'goal_velocity': 0.0, 'goal_iq': 0.0})
print(f"Direct Force mode enabled (spring gains = 0, so this behaves like pure iq control, "
      f"with hardware joint-limit protection). Ramping to {IQ_MAX} A — watch the scale.")

log_t, log_iq_cmd, log_iq_meas, log_pos = [], [], [], []
t0 = time.time()
aborted = False
iq_cmd = IQ_START

while iq_cmd <= IQ_MAX + 1e-9:
    bear.set_goal_iq((MOTOR_ID, iq_cmd))
    time.sleep(STEP_DT)

    pos, _ = read1(bear.get_present_position, MOTOR_ID)
    iq_meas, _ = read1(bear.get_present_iq, MOTOR_ID)
    torque_en, _ = read1(bear.get_torque_enable, MOTOR_ID)
    t = time.time() - t0
    log_t.append(t); log_iq_cmd.append(iq_cmd); log_iq_meas.append(iq_meas); log_pos.append(pos)

    traveled_deg = math.degrees((pos - START_POS) * DIRECTION)
    print(f"  t={t:5.2f}s  iq_cmd={iq_cmd:+.3f} A  iq_meas={iq_meas:+.3f} A  "
          f"pos={pos:+.4f} rad ({traveled_deg:+.1f} deg from start)", end='\r')

    if torque_en == 3:
        print(f"\nHARDWARE JOINT LIMIT TRIPPED — BEAR entered safe damping (torque_enable=3) "
              f"at {traveled_deg:.1f} deg. Disabling to clear the latched error.")
        aborted = True
        bear.set_torque_enable((MOTOR_ID, 0))
        break

    if traveled_deg > ROTATION_DEG + OVERTRAVEL_DEG + POSITION_MARGIN_DEG:
        print(f"\nABORT: shaft traveled {traveled_deg:.1f} deg — past the expected scale "
              f"contact point ({ROTATION_DEG + OVERTRAVEL_DEG:.1f} deg). Scale may be "
              f"missing or mispositioned, or the horn slipped.")
        aborted = True
        bear.set_goal_iq((MOTOR_ID, 0.0))
        break

    iq_cmd += IQ_STEP

print()

if not aborted:
    print(f"Ramp complete at iq={log_iq_cmd[-1]:+.3f} A. Holding {HOLD_TIME_S}s — read the scale now.")
    t_hold0 = time.time()
    while time.time() - t_hold0 < HOLD_TIME_S:
        pos, _ = read1(bear.get_present_position, MOTOR_ID)
        iq_meas, _ = read1(bear.get_present_iq, MOTOR_ID)
        torque_en, _ = read1(bear.get_torque_enable, MOTOR_ID)
        t = time.time() - t0
        log_t.append(t); log_iq_cmd.append(log_iq_cmd[-1]); log_iq_meas.append(iq_meas); log_pos.append(pos)
        print(f"  holding  iq_meas={iq_meas:+.3f} A  pos={pos:+.4f} rad", end='\r')
        if torque_en == 3:
            print("\nHARDWARE JOINT LIMIT TRIPPED during hold — disabling.")
            bear.set_torque_enable((MOTOR_ID, 0))
            break
        time.sleep(0.1)
    print()

    print("Ramping iq back down to 0 A.")
    iq_cmd = log_iq_cmd[-1]
    while iq_cmd > 0:
        iq_cmd = max(0.0, iq_cmd - IQ_STEP)
        bear.set_goal_iq((MOTOR_ID, iq_cmd))
        time.sleep(STEP_DT)

bear.set_goal_iq((MOTOR_ID, 0.0))
time.sleep(0.1)
pos_at_end, _ = read1(bear.get_present_position, MOTOR_ID)
print(f"Ramp done. iq zeroed. Position before retract: {pos_at_end:.4f} rad "
      f"({math.degrees((pos_at_end - START_POS) * DIRECTION):+.1f} deg from start).")

## 8. Retract to start position (Position mode)

Switch to Position mode — a closed loop — and step smoothly back to `START_POS` rather than
just cutting power and hoping the scale (or gravity) pushes the horn back.

In [ ]:
print(f"Retracting to start ({START_POS:.4f} rad) over {RETRACT_TIME_S}s...")

switch_mode(MOTOR_ID, MODE_POSITION, {'goal_position': pos_at_end})

RETRACT_STEPS = 100
for i in range(RETRACT_STEPS):
    goal = pos_at_end + (START_POS - pos_at_end) * (i + 1) / RETRACT_STEPS
    bear.set_goal_position((MOTOR_ID, goal))
    time.sleep(RETRACT_TIME_S / RETRACT_STEPS)

time.sleep(0.3)  # let it settle
pos_final, err = read1(bear.get_present_position, MOTOR_ID)
print(f"Retracted. Final position: {pos_final:.4f} rad (target {START_POS:.4f} rad)")
print(f"Error byte: 0x{err:02x} ({bear.decode_error(err)})")

disable()
print("Motor disabled.")

## 9. Plot the ramp

Commanded vs. measured `iq`, and shaft travel, over the course of the ramp/hold/ramp-down.
Cross-reference against your manually recorded scale readings to build the force-vs-`iq`
curve.

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(8, 5))

ax1.plot(log_t, log_iq_cmd, label='iq commanded', linestyle='--')
ax1.plot(log_t, log_iq_meas, label='iq measured')
ax1.set_ylabel('iq (A)')
ax1.legend()
ax1.grid(True)

travel_deg = [math.degrees((p - START_POS) * DIRECTION) for p in log_pos]
ax2.plot(log_t, travel_deg)
ax2.axhline(ROTATION_DEG + OVERTRAVEL_DEG, color='gray', linestyle=':', label='expected contact')
ax2.set_ylabel('travel from start (deg)')
ax2.set_xlabel('time (s)')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

## Appendix: other modes

The `switch_mode` helper from section 2 works with any of the four modes. These cells are
standalone manual tests (adapted from the base motor-test notebook) — run them independently of
the scale-ramp flow above if you want to exercise Velocity, Position, or a spring-like Direct
Force on this setup. Note that `limit_position_min/max` is still the tight window computed in
section 5 at this point — loosen it first (re-run with a larger `POSITION_MARGIN_DEG`, or set
the limits directly) if these appendix cells need more room to move.

### Velocity mode (mode 1)

In [ ]:
switch_mode(MOTOR_ID, MODE_VELOCITY, {'goal_velocity': 0.0})
print("Enabled in Velocity mode at 0 rad/s.")

TARGET_VEL = -1.0   # rad/s — gentle
RAMP_STEPS = 30

for i in range(RAMP_STEPS):
    bear.set_goal_velocity((MOTOR_ID, TARGET_VEL * (i + 1) / RAMP_STEPS))
    time.sleep(0.02)

vel_samples = []
for _ in range(10):
    v, _ = read1(bear.get_present_velocity, MOTOR_ID)
    vel_samples.append(v)
    time.sleep(0.1)
print(f"Mean held velocity: {sum(vel_samples)/len(vel_samples):+.3f} rad/s (target {TARGET_VEL:+.3f})")

for i in range(RAMP_STEPS):
    bear.set_goal_velocity((MOTOR_ID, TARGET_VEL * (RAMP_STEPS - i - 1) / RAMP_STEPS))
    time.sleep(0.02)

bear.set_goal_velocity((MOTOR_ID, 0.0))
time.sleep(0.2)
disable()
print("Velocity test complete, motor disabled.")

### Position mode (mode 2)

In [ ]:
pos_now, _ = read1(bear.get_present_position, MOTOR_ID)
print(f"Current pos = {pos_now:.4f} rad — using as initial goal.")

switch_mode(MOTOR_ID, MODE_POSITION, {'goal_position': pos_now})
print("Enabled in Position mode, holding current angle.")

TARGET_POS = START_POS
TO_POS_DELTA = TARGET_POS - pos_now
STEPS = 100

for i in range(STEPS):
    goal = pos_now + TO_POS_DELTA * (i + 1) / STEPS
    bear.set_goal_position((MOTOR_ID, goal))
    time.sleep(0.02)

time.sleep(0.3)
pos_final, err = read1(bear.get_present_position, MOTOR_ID)
print(f"Goal      : {TARGET_POS:.4f} rad")
print(f"Reached   : {pos_final:.4f} rad")
print(f"Error byte: 0x{err:02x} ({bear.decode_error(err)})")
disable()

### Direct Force mode, spring-damper behavior (mode 3)

In [ ]:
bear.set_p_gain_force((MOTOR_ID, 5.0))
bear.set_i_gain_force((MOTOR_ID, 0.0))
bear.set_d_gain_force((MOTOR_ID, 0.2))

pos_now, _ = read1(bear.get_present_position, MOTOR_ID)
switch_mode(MOTOR_ID, MODE_FORCE, {
    'goal_position': pos_now,
    'goal_velocity': 0.0,
    'goal_iq':       0.0,
})
print(f"Direct Force enabled, anchored at {pos_now:.4f} rad. Try moving the shaft.")
print("Sampling for 5 s...")

t0 = time.time()
while time.time() - t0 < 5.0:
    p, _  = read1(bear.get_present_position, MOTOR_ID)
    iq, _ = read1(bear.get_present_iq, MOTOR_ID)
    print(f"  t={time.time()-t0:4.1f}s  pos={p:+.4f} rad  iq={iq:+.3f} A", end='\r')
    time.sleep(0.1)
print()

disable()
print("Direct Force test complete, motor disabled.")

# Reset force gains to 0 so section 7's pure-iq behavior holds if you re-run it.
bear.set_p_gain_force((MOTOR_ID, 0.0))
bear.set_d_gain_force((MOTOR_ID, 0.0))

## EMERGENCY DISABLE

Run this cell at any time to cut current. Safe to run repeatedly.

In [ ]:
try:
    bear.set_torque_enable((MOTOR_ID, 0))
    print("Disabled.")
except Exception as e:
    print(f"Disable attempt failed: {e!r}. Pull power if motor is still moving.")

## Cleanup

Disable and close the serial port.

In [ ]:
bear.set_torque_enable((MOTOR_ID, 0))
bear.close_port()
print("Disabled and port closed.")